# 🧩 Mini-Lab: Modules and Imports — Building the Triage Package

**Module: The Python That Runs Your Pipeline** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Explain** the difference between a Python module, a package, and a script
2. **Organize** pure functions into separate `.py` modules grouped inside a package
3. **Import** your package from any notebook or script without path hacks

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Python Module & Package | Files (modules) and folders with __init__.py (packages) make code importable from anywhere, so your pipeline logic is reusable, testable, and easier to run outside notebooks. |
| Pure Function | A pure function takes inputs only through its parameters and produces results only via its return value, printing and side effects are the caller's job. |


## Why Packages Matter

In the previous mini-labs we extracted pure functions and added type hints. But those functions still live **inside notebook cells**. If you want to use `score_documents` in a different notebook, a FastAPI endpoint, or a CLI script, you'd have to copy-paste it.

The fix: move functions into **`.py` files** (modules) and group them inside a **package** (a folder with `__init__.py`). Then any code anywhere can simply `import` them.

| Term | What It Is | Example |
|------|------------|---------|
| **Module** | A single `.py` file containing functions/classes | `scoring.py` |
| **Package** | A folder with an `__init__.py` file | `triage/` |
| **`__init__.py`** | Marks a folder as a package; can re-export names | `from .scoring import score_documents` |

## What We'll Build

We'll create a tiny package called **`triage`** with two modules:

```
triage/
├── __init__.py      ← makes it a package; re-exports public names
├── scoring.py       ← score_documents (from mini-extract-pure-function)
└── formatting.py    ← format_sources  (from mini-add-type-hints)
```

After creating these files, we'll import and use them just like any installed library.

## Step 1: Create the Package Folder

We use Python's `pathlib` to create the folder. This works on both Mac and Windows.

In [1]:
from pathlib import Path

pkg_dir = Path("triage")
pkg_dir.mkdir(exist_ok=True)
print(f"Created package folder: {pkg_dir.resolve()}")

Created package folder: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-build-triage-package\triage


## Step 2: Write the Modules

We'll use `%%writefile` to create each `.py` file directly from the notebook. These are the same pure functions from the previous mini-labs — now living in their own files.

### `scoring.py` — document scoring logic

In [2]:
%%writefile triage/scoring.py
"""Document scoring by keyword overlap."""


def score_documents(
    query: str, documents: list[str], threshold: float = 0.3
) -> list[dict]:
    """Score documents by keyword overlap with the query.

    Returns a sorted list of dicts with 'text' and 'score' keys,
    including only documents at or above the threshold.
    """
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append({"text": doc, "score": round(overlap, 2)})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

Writing triage/scoring.py


### `formatting.py` — source formatting logic

In [3]:
%%writefile triage/formatting.py
"""Formatting utilities for retrieved documents."""


def format_sources(
    docs: list[dict[str, str]], max_sources: int = 3
) -> str:
    """Format retrieved documents into a numbered source list."""
    lines = []
    for i, doc in enumerate(docs[:max_sources], 1):
        lines.append(f"{i}. {doc['text']}")
    return "\n".join(lines)

Writing triage/formatting.py


## Step 3: Write `__init__.py`

The `__init__.py` file does two jobs:

1. **Marks the folder as a package** so Python recognizes it for imports.
2. **Re-exports public names** so users can write `from triage import score_documents` instead of the longer `from triage.scoring import score_documents`.

In [4]:
%%writefile triage/__init__.py
"""Triage package — document scoring and formatting utilities."""

from .scoring import score_documents
from .formatting import format_sources

__all__ = ["score_documents", "format_sources"]

Writing triage/__init__.py


Let's verify the file structure looks right:

In [5]:
for f in sorted(pkg_dir.rglob("*.py")):
    print(f"  {f}")

  triage\__init__.py
  triage\formatting.py
  triage\scoring.py


## Step 4: Import and Use the Package

Now we import from `triage` just like any library. Because the notebook is in the same directory as the `triage/` folder, Python finds it automatically — no `sys.path` hacks needed.

In [6]:
from triage import score_documents, format_sources

# Sample data
docs = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]

# Score
results = score_documents("How do transformers handle long sequences?", docs)
print("Scored results:")
for r in results:
    print(f"  [{r['score']}] {r['text']}")

# Format
print("\nFormatted sources:")
print(format_sources(results))

Scored results:

Formatted sources:



## Three Ways to Import

Because `__init__.py` re-exports the public names, users have flexibility in how they import. All three styles below work:

In [7]:
# Style 1: Import specific names from the package (recommended)
from triage import score_documents

# Style 2: Import a specific module, then use dotted access
from triage import scoring
result = scoring.score_documents("attention", docs, threshold=0.0)

# Style 3: Import the whole package
import triage
result = triage.score_documents("attention", docs, threshold=0.0)

print("All three import styles work ✓")

All three import styles work ✓


## Key Rules Recap

| Rule | Details |
|------|---------|
| **One job per module** | `scoring.py` does scoring, `formatting.py` does formatting — don't dump everything in one file |
| **`__init__.py` re-exports** | Import from sub-modules and expose them at the package level for convenience |
| **Pure functions are ideal module contents** | They have no hidden state, so they work the same whether called from a notebook, a script, or a web server |
| **No `sys.path` hacks** | If the package folder is next to your notebook/script, imports just work |

## Cleanup

Remove the `triage/` folder so it doesn't clutter your workspace (optional — run this only if you want to clean up).

In [8]:
import shutil

shutil.rmtree("triage", ignore_errors=True)
print("Cleaned up triage/ folder ✓")

Cleaned up triage/ folder ✓


## 🎯 Summary

### Key Takeaways

1. **A module is a `.py` file** — move pure functions out of notebook cells and into modules so they can be imported anywhere
2. **A package is a folder with `__init__.py`** — it groups related modules and can re-export names for convenient imports
3. **`__init__.py` controls the public API** — use `from .module import func` to let users write `from package import func` directly
4. **Pure functions are the ideal package contents** — no globals, no side effects, so they work identically whether called from a notebook, a script, or a server